# Practica M29: Obtención de datos de páginas web y APIs con Python
#### Objetivo: aprender a utilizar requests para consultar una página web, extraer información con BeautifulSoup, organizar los datos en listas y diccionarios y consultar una API pública.
#### Trabajaré con estos sitios:
#### - Books to Scrape: https://books.toscrape.com/ (sitio de prueba para hacer web scraping).
#### - JSONPlaceholder: https://jsonplaceholder.typicode.com/users (API pública con datos de ejemplo).

## Paso 1. Instalar los paquetes
#### Primero instalo los paquetes que utilizaré. requests permite consultar páginas web y APIs, mientras que BeautifulSoup me ayuda a leer el HTML.

In [ ]:
%pip install requests beautifulsoup4

## Paso 2. Importar las librerías y cambiar de directorio
#### También usaré os para elegir dónde guardar los archivos y csv para exportar los resultados.

In [ ]:
import os
import csv
import requests
from bs4 import BeautifulSoup

# Cambio el directorio de trabajo a Downloads
os.chdir("/Users/santiago/Downloads")
print("Directorio actual:", os.getcwd())

## Paso 3. Consultar una página web con requests
#### Con requests.get() hago la solicitud. status_code me permite revisar la respuesta y response.text contiene el código HTML.
#### Para no llenar todo el Notebook, mostraré los primeros 1,500 caracteres. Si quiero imprimir el HTML completo, puedo cambiar print(html[:1500]) por print(html).

In [ ]:
url = "https://books.toscrape.com/"
respuesta = requests.get(url, timeout=20)
respuesta.raise_for_status()

html = respuesta.text
print("Código de respuesta:", respuesta.status_code)
print("Primeros 1500 caracteres del HTML:")
print(html[:1500])

## Paso 4. Obtener información con BeautifulSoup
#### Ahora convierto el HTML en un objeto que Python pueda recorrer. Buscaré los libros de la primera página y obtendré su título y su precio. La página contiene 20 libros por página; aquí no es necesario descargar todas sus páginas.

In [ ]:
sopa = BeautifulSoup(html, "html.parser")

# Busco las tarjetas que contienen los libros
productos = sopa.find_all("article", class_="product_pod")

# Guardo los datos en una lista de diccionarios
libros = []

for producto in productos:
    titulo = producto.find("h3").find("a")["title"]
    precio = producto.find("p", class_="price_color").get_text(strip=True)

    libros.append({"Titulo": titulo, "Precio": precio})

print("Libros encontrados:", len(libros))
print("Primeros 5 libros:")
for libro in libros[:5]:
    print(libro)

## Paso 5. Crear un listado simple y ordenado
#### Utilizaré sorted() para ordenar los libros alfabéticamente por título. Después usaré un ciclo for para numerarlos.

In [ ]:
libros_ordenados = sorted(libros, key=lambda libro: libro["Titulo"].lower())

for numero, libro in enumerate(libros_ordenados, start=1):
    print(numero, "-", libro["Titulo"], "-", libro["Precio"])

## Paso 6. Guardar el listado de libros en un CSV
#### Así puedo utilizar los datos fuera de Python y abrirlos posteriormente en Excel.

In [ ]:
with open("libros_web_ordenados.csv", "w", newline="", encoding="utf-8-sig") as archivo:
    escritor = csv.DictWriter(archivo, fieldnames=["Titulo", "Precio"])
    escritor.writeheader()
    escritor.writerows(libros_ordenados)

print("Archivo guardado: libros_web_ordenados.csv")

## Paso 7. Consultar una API con requests
#### Ahora utilizaré JSONPlaceholder, que devuelve datos de usuarios de ejemplo en formato JSON. Con respuesta_api.json() convierto la respuesta directamente en una estructura de Python.

In [ ]:
url_api = "https://jsonplaceholder.typicode.com/users"
respuesta_api = requests.get(url_api, timeout=20)
respuesta_api.raise_for_status()

usuarios = respuesta_api.json()

print("Código de respuesta:", respuesta_api.status_code)
print("Tipo de estructura:", type(usuarios))
print("Usuarios recibidos:", len(usuarios))
print("Primer usuario:")
print(usuarios[0])

## Paso 8. Leer listas y diccionarios de la API
#### Los usuarios vienen en una lista de diccionarios. Dentro de cada usuario también hay diccionarios anidados, como address. Seleccionaré el nombre, correo y ciudad.

In [ ]:
usuarios_resumidos = []

for usuario in usuarios:
    datos = {
        "ID": usuario["id"],
        "Nombre": usuario["name"],
        "Correo": usuario["email"],
        "Ciudad": usuario["address"]["city"]
    }
    usuarios_resumidos.append(datos)

for usuario in usuarios_resumidos[:5]:
    print(usuario)

## Paso 9. Ordenar los usuarios alfabéticamente
#### Igual que con los libros, utilizo sorted() para ordenar una lista por el nombre de cada usuario.

In [ ]:
usuarios_ordenados = sorted(usuarios_resumidos, key=lambda usuario: usuario["Nombre"].lower())

for numero, usuario in enumerate(usuarios_ordenados, start=1):
    print(numero, "-", usuario["Nombre"], "-", usuario["Ciudad"])

## Paso 10. Guardar el resultado de la API en un CSV
#### Finalmente exportaré los datos seleccionados para tener un segundo archivo con la información obtenida de Internet.

In [ ]:
with open("usuarios_api_ordenados.csv", "w", newline="", encoding="utf-8-sig") as archivo:
    escritor = csv.DictWriter(archivo, fieldnames=["ID", "Nombre", "Correo", "Ciudad"])
    escritor.writeheader()
    escritor.writerows(usuarios_ordenados)

print("Archivo guardado: usuarios_api_ordenados.csv")